<!--encabezado-grupo-->
> **Archivo:** `notebooks/tomas_paso_a_paso.ipynb`  
> **Objetivo:** Notebook: tomas_paso_a_paso  
> **Grupo del curso:** Fabio Durán - Gustavo Reyes - Ezequiel Muñoz  
> **Creación:** 20261110 1952 · **Última modificación:** 20261111 1410
<!--/encabezado-grupo-->

# Agente de Tomás (Maestro Reiki) · paso a paso

Tres pasos: **(1) herramientas**, **(2) el agente por A2A**, **(3) consumo**.

Pensado para el **segundo computador (Windows)**. Pasos 2–3 requieren `langgraph dev` + `GOOGLE_API_KEY`.

## 1 · Herramientas (sin servidor ni API key)

In [ ]:
from src.agente_tomas.herramientas import principios_reiki, posiciones_manos, info_simbolo
print(principios_reiki.invoke({}))
print(posiciones_manos.invoke({'zona':'espalda'}))
print(info_simbolo.invoke({'nombre':'Cho Ku Rei'}))

## 2 · El agente por A2A nativo

Con `langgraph dev` arriba, el servidor publica `agente_tomas`.

In [ ]:
import httpx
from dotenv import load_dotenv; load_dotenv()
BASE='http://127.0.0.1:2024'
aid=httpx.post(f'{BASE}/assistants/search', json={'graph_id':'agente_tomas'}, timeout=10).json()[0]['assistant_id']
card=httpx.get(f'{BASE}/.well-known/agent-card.json', params={'assistant_id':aid}, timeout=10).json()
print('nombre:', card.get('name')); print('descripción:', card.get('description'))

## 3 · Consumo

Preguntas por `message/send`: el agente usa sus herramientas, no da consejo médico y rechaza lo ajeno.

In [ ]:
import uuid, httpx
def preguntar(t):
    c={'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','messageId':uuid.uuid4().hex,'contextId':str(uuid.uuid4()),'parts':[{'kind':'text','text':t}]}}}
    r=httpx.post(f'{BASE}/a2a/{aid}', json=c, timeout=90).json(); print('>',t); print(r.get('result',r),'\n')
preguntar('¿Cuáles son los cinco principios del Reiki?')
preguntar('¿Qué posiciones de manos uso para la espalda?')
preguntar('¿Qué antibiótico tomo para la fiebre?')  # salud: deriva a un profesional